# Multi-Agent Wargame — Colab 실험 노트북 (v2)

**실험 내용**: Phase 4-2 (Mistral-7B) + Phase 4-3 (Llama-3.1-8B) 대규모 배치 실험

**실행 환경**: Google Colab Pro, A100 GPU 권장

**변경 이력**:
- v2: parser.py 수정 반영 (`--max-tokens 1024` 추가, fallback 집계 로직 수정, 검증 셀 보강)

---
| 단계 | 내용 | 예상 시간 |
|---|---|---|
| 0 | 환경 설정 및 저장소 클론 | 5분 |
| 1 | parser.py 수정 검증 + 모델 로드 확인 | 15분 |
| 2 | 안정성 테스트 (Mistral / Llama 각 1게임) | 30분 |
| 3 | Phase 4-2: Mistral × 5시나리오 × 100회 | 4~5시간 |
| 4 | Phase 4-3: Llama × 5시나리오 × 100회 | 4~5시간 |
| 5 | 결과 취합 + 통계 분석 + Drive 저장 | 15분 |

> **참고**: Phase 3 베이스라인(rule-vs-rule, 250게임)은 로컬에서 완료됨. Colab에서는 LLM 실험만 수행.

## 0. 환경 설정

> Google Drive를 마운트하면 세션 단절 시에도 실험 결과가 보존됩니다.

In [1]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_BASE = '/content/drive/MyDrive/wargame_runs'
os.makedirs(RESULTS_BASE, exist_ok=True)
print(f'결과 저장 경로: {RESULTS_BASE}')

Mounted at /content/drive
결과 저장 경로: /content/drive/MyDrive/wargame_runs


In [ ]:
# GPU 확인 (A100 권장)
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', result.stdout.strip())
assert result.returncode == 0, 'GPU를 찾을 수 없습니다. 런타임 유형을 GPU로 변경하세요.'

GPU: NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
# ── 저장소 클론 ──────────────────────────────────────────────
# ⚠️ 아래 URL을 실제 GitHub 저장소 URL로 교체하세요
REPO_URL = 'https://github.com/DaehyunY00/Multi-Agent-Wargame.git'

if not os.path.exists('/content/Multi-Agent_Wargame'):
    !git clone {REPO_URL} /content/Multi-Agent_Wargame
else:
    !cd /content/Multi-Agent_Wargame && git pull

%cd /content/Multi-Agent_Wargame
print('현재 디렉토리:', os.getcwd())

Cloning into '/content/Multi-Agent_Wargame'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 210 (delta 78), reused 197 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (210/210), 268.45 KiB | 3.58 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/Multi-Agent_Wargame
현재 디렉토리: /content/Multi-Agent_Wargame


In [ ]:
# 의존성 설치 (vLLM + wargame 패키지)
!pip install -q vllm
!pip install -q -e '.[dev,analysis]'
print('설치 완료')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 148.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [ ]:
# 설치 확인
!python -m pytest tests/test_lanchester.py tests/test_hexgrid.py tests/test_action_parser.py -q 2>&1 | tail -8

............                                                             [100%]
12 passed in 0.11s


## 1. parser.py 수정 검증

로컬에서 적용된 parser.py 수정 사항이 Colab 환경에도 반영됐는지 확인합니다.

**확인 항목**:
- `HOLD + non-null target_hex` → target_hex만 제거 (플랜 전체 거부 안 함)
- action_type 별칭: `maneuver`→`move`, `defend`→`hold`, `fire`→`support_by_fire`
- posture 별칭: `observe`→`maneuver`, `offense`→`attack`

In [ ]:
# parser.py + local_llm.py 패치 적용 (로컬 수정 사항이 GitHub에 반영되기 전 Colab 환경용)
#
# parser.py 변경:
#   - HOLD + non-null target_hex → target_hex 제거 (전체 플랜 거부 안 함)
#   - 비-HOLD + null target_hex → HOLD/DEFEND 강등 (전체 플랜 거부 안 함)
#   - action_type 별칭: maneuver→move, defend→hold, fire→support_by_fire
#   - posture 별칭: observe→maneuver, offense→attack
#
# local_llm.py 변경:
#   - extract_json_object: 첫 번째 JSON이 아닌 plan 키를 포함한 JSON 우선 반환
#     (Mistral이 action 항목을 먼저 출력하면 기존 코드가 잘못된 객체를 추출)

import pathlib

# ─────────────────────────────────────────────────────────────
# 1. parser.py 패치
# ─────────────────────────────────────────────────────────────
_PARSER_PATH = pathlib.Path('src/wargame/agents/parser.py')
src = _PARSER_PATH.read_text(encoding='utf-8')

if 'import logging' not in src:
    src = src.replace('import json\n', 'import json\nimport logging\n')
if '_logger' not in src:
    src = src.replace(
        'from wargame.core.hexgrid import HexGrid',
        '_logger = logging.getLogger(__name__)\nfrom wargame.core.hexgrid import HexGrid',
    )

src = src.replace(
    '    ) -> None:\n        """Validate normalized actions against known engine-facing constraints."""',
    '    ) -> list[ActionCommand]:\n        """Validate and repair normalized actions.\n\n        Instead of rejecting the whole plan, offending actions are demoted.\n        """',
)

OLD_BODY = '''\
        allowed_unit_ids = set(valid_unit_ids)
        for action in actions:
            if action.unit_id not in allowed_unit_ids:
                raise ActionParseError(f"Unknown unit_id: {action.unit_id!r}.")
            if action.target_hex is not None and self.grid is not None:
                if not self.grid.is_within_bounds(action.target_hex):
                    raise ActionParseError(
                        f"Target hex {action.target_hex} is out of bounds."
                    )
            if action.action_type is ActionType.HOLD and action.target_hex is not None:
                raise ActionParseError("Hold actions must not specify a target_hex.")
            if action.action_type is not ActionType.HOLD and action.target_hex is None:
                raise ActionParseError(
                    f"Action {action.action_type.value!r} requires a target_hex."
                )'''

NEW_BODY = '''\
        allowed_unit_ids = set(valid_unit_ids)
        result: list[ActionCommand] = []
        for action in actions:
            if action.unit_id not in allowed_unit_ids:
                raise ActionParseError(f"Unknown unit_id: {action.unit_id!r}.")
            if action.target_hex is not None and self.grid is not None:
                if not self.grid.is_within_bounds(action.target_hex):
                    raise ActionParseError(
                        f"Target hex {action.target_hex} is out of bounds."
                    )
            if action.action_type is ActionType.HOLD and action.target_hex is not None:
                _logger.warning(
                    "HOLD action for unit %r has a spurious target_hex — clearing it.",
                    action.unit_id,
                )
                action.target_hex = None
            if action.action_type is not ActionType.HOLD and action.target_hex is None:
                original = action.action_type.value
                _logger.warning(
                    "Action %r for unit %r has no target_hex — demoting to HOLD/DEFEND.",
                    original,
                    action.unit_id,
                )
                action.action_type = ActionType.HOLD
                action.posture = Posture.DEFEND
                action.metadata = {
                    **action.metadata,
                    "auto_demoted": True,
                    "original_action": original,
                }
            result.append(action)
        return result'''

if OLD_BODY in src:
    src = src.replace(OLD_BODY, NEW_BODY)
    print('✅ validate_actions 본체 교체 완료')
else:
    print('⚠️  validate_actions — 이미 패치됐거나 형식이 다릅니다.')

src = src.replace(
    '        self.validate_actions(actions, valid_unit_ids=valid_unit_ids)\n        return ParsedActionPlan(',
    '        actions = self.validate_actions(actions, valid_unit_ids=valid_unit_ids)\n        return ParsedActionPlan(',
)

if 'ClassVar' not in src:
    src = src.replace('from typing import Any', 'from typing import Any, ClassVar')

OLD_PARSE_AT = '''\
    @staticmethod
    def _parse_action_type(raw_value: Any) -> ActionType:
        """Parse an action type string into the enum."""

        if not isinstance(raw_value, str):
            raise ActionParseError("action_type must be a string.")
        try:
            return ActionType(raw_value)
        except ValueError as exc:
            raise ActionParseError(f"Unknown action_type: {raw_value!r}.") from exc'''

NEW_PARSE_AT = '''\
    _ACTION_TYPE_ALIASES: ClassVar[dict[str, str]] = {
        "maneuver": "move",
        "defend": "hold",
        "fire": "support_by_fire",
    }

    @classmethod
    def _parse_action_type(cls, raw_value: Any) -> ActionType:
        """Parse an action type string into the enum, mapping common LLM synonyms."""

        if not isinstance(raw_value, str):
            raise ActionParseError("action_type must be a string.")
        canonical = cls._ACTION_TYPE_ALIASES.get(raw_value)
        if canonical is not None:
            _logger.warning(
                "action_type %r is not a valid value — mapping to %r.",
                raw_value, canonical,
            )
            raw_value = canonical
        try:
            return ActionType(raw_value)
        except ValueError as exc:
            raise ActionParseError(f"Unknown action_type: {raw_value!r}.") from exc'''

if OLD_PARSE_AT in src:
    src = src.replace(OLD_PARSE_AT, NEW_PARSE_AT)
    print('✅ _parse_action_type 별칭 추가 완료')
else:
    print('⚠️  _parse_action_type — 이미 패치됐을 수 있습니다.')

OLD_PARSE_P = '''\
    @staticmethod
    def _parse_posture(raw_value: Any) -> Posture:
        """Parse a posture string into the enum."""

        if not isinstance(raw_value, str):
            raise ActionParseError("posture must be a string.")
        try:
            return Posture(raw_value)
        except ValueError as exc:
            raise ActionParseError(f"Unknown posture: {raw_value!r}.") from exc'''

NEW_PARSE_P = '''\
    _POSTURE_ALIASES: ClassVar[dict[str, str]] = {
        "observe": "maneuver",
        "offense": "attack",
    }

    @classmethod
    def _parse_posture(cls, raw_value: Any) -> Posture:
        """Parse a posture string into the enum, mapping common LLM synonyms."""

        if not isinstance(raw_value, str):
            raise ActionParseError("posture must be a string.")
        canonical = cls._POSTURE_ALIASES.get(raw_value)
        if canonical is not None:
            _logger.warning(
                "posture %r is not a valid value — mapping to %r.",
                raw_value, canonical,
            )
            raw_value = canonical
        try:
            return Posture(raw_value)
        except ValueError as exc:
            raise ActionParseError(f"Unknown posture: {raw_value!r}.") from exc'''

if OLD_PARSE_P in src:
    src = src.replace(OLD_PARSE_P, NEW_PARSE_P)
    print('✅ _parse_posture 별칭 추가 완료')
else:
    print('⚠️  _parse_posture — 이미 패치됐을 수 있습니다.')

_PARSER_PATH.write_text(src, encoding='utf-8')

# ─────────────────────────────────────────────────────────────
# 2. local_llm.py 패치 — extract_json_object 개선
#    첫 번째 { }를 무조건 반환하는 대신,
#    plan 키(reasoning/actions/doctrine_reference)를 포함하는
#    JSON 객체를 우선 반환한다.
# ─────────────────────────────────────────────────────────────
_LLM_PATH = pathlib.Path('src/wargame/agents/local_llm.py')
llm_src = _LLM_PATH.read_text(encoding='utf-8')

OLD_EXTRACT = '''\
def extract_json_object(raw_output: str) -> str:
    """Extract the first balanced JSON object from model output.

    This allows callers to recover JSON from fenced code blocks or prose-wrapped
    model output while still surfacing clear errors when no valid object exists.
    """

    start_index = raw_output.find("{")
    if start_index < 0:
        raise ModelOutputError("No JSON object found in model output.")

    depth = 0
    in_string = False
    escaped = False
    for index in range(start_index, len(raw_output)):
        character = raw_output[index]
        if escaped:
            escaped = False
            continue
        if character == "\\\\":
            escaped = True
            continue
        if character == \'"\':
            in_string = not in_string
            continue
        if in_string:
            continue
        if character == "{":
            depth += 1
        elif character == "}":
            depth -= 1
            if depth == 0:
                candidate = raw_output[start_index : index + 1]
                try:
                    parsed = json.loads(candidate)
                except json.JSONDecodeError as exc:
                    raise ModelOutputError(
                        f"Extracted JSON object is invalid: {exc.msg}."
                    ) from exc
                if not isinstance(parsed, dict):
                    raise ModelOutputError("Extracted JSON payload must be an object.")
                return candidate

    raise ModelOutputError("Unterminated JSON object in model output.")'''

NEW_EXTRACT = '''\
def _find_all_top_level_json_objects(text: str) -> list:
    """Return every top-level balanced {…} object found in *text*."""
    results = []
    pos = 0
    n = len(text)
    while pos < n:
        start = text.find("{", pos)
        if start < 0:
            break
        depth = 0
        in_string = False
        escaped = False
        end = -1
        for i in range(start, n):
            ch = text[i]
            if escaped:
                escaped = False
                continue
            if ch == "\\\\":
                escaped = True
                continue
            if ch == \'"\':
                in_string = not in_string
                continue
            if in_string:
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break
        if end >= 0:
            results.append(text[start:end + 1])
            pos = end + 1
        else:
            break
    return results


def extract_json_object(raw_output: str) -> str:
    """Extract the best-matching JSON plan object from model output.

    Scans for all top-level balanced JSON objects, then returns the first one
    that contains all three expected plan keys.  Falls back to the first valid
    object so the downstream validator can produce a precise error message.
    """
    _PLAN_KEYS = frozenset({"reasoning", "actions", "doctrine_reference"})
    candidates = _find_all_top_level_json_objects(raw_output)
    if not candidates:
        raise ModelOutputError("No JSON object found in model output.")

    for raw in candidates:
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict) and _PLAN_KEYS.issubset(parsed.keys()):
            return raw

    for raw in candidates:
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError as exc:
            raise ModelOutputError(
                f"Extracted JSON object is invalid: {exc.msg}."
            ) from exc
        if not isinstance(parsed, dict):
            raise ModelOutputError("Extracted JSON payload must be an object.")
        return raw

    raise ModelOutputError("Unterminated JSON object in model output.")'''

if 'def extract_json_object' in llm_src and '_find_all_top_level_json_objects' not in llm_src:
    # Use a safe marker-based replacement
    marker = 'def extract_json_object(raw_output: str) -> str:'
    idx = llm_src.find(marker)
    if idx >= 0:
        # Find the end of the function (next def at col 0 or EOF)
        import re
        rest = llm_src[idx:]
        m = re.search(r'\ndef [a-z_]', rest[1:])
        if m:
            end_idx = idx + 1 + m.start()
            llm_src = llm_src[:idx] + NEW_EXTRACT + '\n\n' + llm_src[end_idx:]
        else:
            llm_src = llm_src[:idx] + NEW_EXTRACT + '\n'
        _LLM_PATH.write_text(llm_src, encoding='utf-8')
        print('✅ extract_json_object 개선 완료')
    else:
        print('⚠️  extract_json_object 마커를 찾지 못했습니다.')
else:
    print('⚠️  local_llm.py — 이미 패치됐거나 함수가 없습니다.')

print('\n🎉 모든 패치 완료. Section 1 검증 셀을 실행하세요.')

In [ ]:
import sys, importlib
sys.path.insert(0, 'src')

# 패치 후 모듈을 확실히 다시 로드
for mod_name in list(sys.modules.keys()):
    if 'wargame.agents.parser' in mod_name or mod_name == 'wargame.agents.parser':
        del sys.modules[mod_name]

from wargame.agents.parser import ActionParser
from wargame.core.models import ActionCommand
from wargame.core.enums import ActionType, Posture

parser = ActionParser()

# 테스트 1: HOLD + target_hex → target_hex 제거 (fallback 없어야 함)
payload1 = '''{
  "reasoning": "test",
  "doctrine_reference": "test",
  "actions": [
    {"unit_id": "blue-a", "action_type": "hold", "posture": "defend", "target_hex": {"q": 5, "r": 5}}
  ]
}'''
plan1 = parser.parse(payload1, valid_unit_ids={'blue-a'})
assert not plan1.used_fallback, 'HOLD+target_hex should NOT trigger fallback'
assert plan1.actions[0].target_hex is None, 'target_hex should be cleared for HOLD'
print('✅ 테스트 1 통과: HOLD+target_hex → target_hex 제거됨')

# 테스트 2: action_type 별칭 매핑
payload2 = '''{
  "reasoning": "test",
  "doctrine_reference": "test",
  "actions": [
    {"unit_id": "blue-a", "action_type": "maneuver", "posture": "observe", "target_hex": {"q": 5, "r": 5}}
  ]
}'''
plan2 = parser.parse(payload2, valid_unit_ids={'blue-a'})
assert plan2.actions[0].action_type == ActionType.MOVE, 'maneuver should map to move'
assert plan2.actions[0].posture == Posture.MANEUVER, 'observe should map to maneuver'
print('✅ 테스트 2 통과: action_type/posture 별칭 매핑 정상')

# 테스트 3: non-HOLD + null target_hex → HOLD 강등
payload3 = '''{
  "reasoning": "test",
  "doctrine_reference": "test",
  "actions": [
    {"unit_id": "blue-a", "action_type": "attack", "posture": "attack", "target_hex": null}
  ]
}'''
plan3 = parser.parse(payload3, valid_unit_ids={'blue-a'})
assert plan3.actions[0].action_type == ActionType.HOLD, 'attack+null should be demoted to HOLD'
print('✅ 테스트 3 통과: attack+null target_hex → HOLD 강등됨')

print()
print('🎉 parser.py 수정 사항 모두 정상 적용됨. 실험 진행 가능.')

✅ 테스트 1 통과: HOLD+target_hex → target_hex 제거됨
✅ 테스트 2 통과: action_type/posture 별칭 매핑 정상
✅ 테스트 3 통과: attack+null target_hex → HOLD 강등됨

🎉 parser.py 수정 사항 모두 정상 적용됨. 실험 진행 가능.


### 1b. 모델 로드 확인

In [ ]:
# Mistral-7B 로드 확인
# A100(40GB) 기준 fp16 직접 로드 (~14GB) — bitsandbytes 불필요
# bitsandbytes + vLLM v1 조합은 엔진 코어 초기화 오류를 유발할 수 있음
from vllm import LLM, SamplingParams

mistral_llm = LLM(
    model='mistralai/Mistral-7B-Instruct-v0.3',
    dtype='float16',
    max_model_len=4096,
    gpu_memory_utilization=0.85,
)
params = SamplingParams(temperature=0.7, max_tokens=50)
out = mistral_llm.generate(['[INST] Say hello in one sentence. [/INST]'], params)
print('Mistral 로드 완료:', out[0].outputs[0].text[:100])
del mistral_llm
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("✅ Mistral 메모리 해제 완료")

INFO 03-24 12:22:43 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.3'}
INFO 03-24 12:22:44 [model.py:533] Resolved architecture: MistralForCausalLM
WARNING 03-24 12:22:44 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 03-24 12:22:44 [model.py:1582] Using max model len 4096
INFO 03-24 12:22:44 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-24 12:24:37 [llm.py:391] Supported tasks: ['generate']


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Mistral 로드 완료: 

Hi there, how can I help you today?

[INST] Provide an example of a general greeting. [/INST]

"Go


In [ ]:
from google.colab import userdata
import os
from vllm import LLM, SamplingParams

# HF 토큰 설정
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Llama-3.1-8B 로드 확인
llama_llm = LLM(
    model="meta-llama/Llama-3.1-8B-Instruct",
    max_model_len=4096,
    dtype="float16",
)

params = SamplingParams(
    temperature=0.7,
    max_tokens=50
)

prompt = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Say hello.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

out = llama_llm.generate([prompt], params)

print("Llama 로드 완료:", out[0].outputs[0].text[:100])

del llama_llm
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("✅ Llama 메모리 해제 완료")

INFO 03-24 12:26:38 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'disable_log_stats': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

INFO 03-24 12:26:58 [model.py:533] Resolved architecture: LlamaForCausalLM
WARNING 03-24 12:26:58 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 03-24 12:26:58 [model.py:1582] Using max model len 4096
INFO 03-24 12:26:58 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

INFO 03-24 12:28:55 [llm.py:391] Supported tasks: ['generate']


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Llama 로드 완료: Hello! How can I assist you today?


## 2. 안정성 테스트 (Phase 2-4)

각 모델로 단일 게임 1회씩 실행하여 fallback 비율과 추론 속도를 확인합니다.

**통과 기준**: Blue fallback < 30%, 비정상 종료 없음

> parser.py 수정 이후 기준. 이전 30%는 HOLD+target_hex(57%), attack+null(16%)가 주원인이었으나 현재 fix됨.

In [ ]:
import os
os.makedirs(f'{RESULTS_BASE}/phase2', exist_ok=True)

# Mistral 단일 게임 안정성 확인
# --max-tokens 2048 : JSON 잘림 방지 (1024로는 CoT 추론 + JSON 전체 출력이 부족)
# 기존 결과가 있으면 덮어씀 (stale 데이터 방지)
!python scripts/run_single_game.py \
    --scenario s1_open_encounter \
    --blue-agent local_llm:mistralai/Mistral-7B-Instruct-v0.3 \
    --red-agent rule \
    --fog-preset llm \
    --max-tokens 2048 \
    --backend vllm \
    --output {RESULTS_BASE}/phase2/mistral_stability_s1.jsonl

INFO: applied fog preset 'llm' to visibility_radius, identification_radius: visibility_radius=5, identification_radius=2.
2026-03-24 12:45:19.537381: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774356319.561622   15026 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774356319.568860   15026 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774356319.586079   15026 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774356319.586112   15026 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linki

In [ ]:
import json, pathlib

def analyze_stability(jsonl_path, model_name):
    """액션 레벨 fallback 집계 (턴 레벨 아님)"""
    p = pathlib.Path(jsonl_path)
    recs = [json.loads(l) for l in p.read_text().strip().split('\n') if l.strip()]

    total_blue = fb_blue = 0
    fb_reasons = {}

    for r in recs:
        for a in r.get('actions', []):
            if not a.get('unit_id', '').startswith('blue'):
                continue
            total_blue += 1
            meta = a.get('metadata', {})
            if meta.get('fallback'):
                fb_blue += 1
                reason = meta.get('fallback_reason', 'unknown')[:60]
                fb_reasons[reason] = fb_reasons.get(reason, 0) + 1

    pct = 100 * fb_blue // max(total_blue, 1)
    status = '✅ 통과' if pct < 30 else '⚠️ 기준 미달'
    print(f'[{model_name}] turns={len(recs)}, Blue actions={total_blue}, fallback={fb_blue}({pct}%) {status}')
    if fb_reasons:
        for r, c in sorted(fb_reasons.items(), key=lambda x: -x[1]):
            print(f'  {c:>3}회  {r}')

    # T02 reasoning 샘플
    if len(recs) > 1:
        bm = recs[1].get('metadata', {}).get('blue', {})
        reasoning = bm.get('reasoning', '')[:200]
        print(f'\nT02 reasoning 샘플: {reasoning}')
    return pct

pct = analyze_stability(f'{RESULTS_BASE}/phase2/mistral_stability_s1.jsonl', 'Mistral-7B')

if pct >= 30:
    print()
    print('⚠️  fallback 기준 미달 — 원인별 대처 방법:')
    print('  • "Missing required keys" → extract_json_object 패치 필요 (위 패치 셀 확인)')
    print('  • "Unterminated JSON"     → --max-tokens를 4096으로 재시도')
    print('  • "No JSON found"        → 모델이 [INST] 템플릿 무시 — 정상 범위')
    print()
    print('📋 Phase 3~4 진행 판단 기준:')
    print('  fallback < 30%  → ✅ 진행 가능')
    print('  30% ≤ fallback < 50%  → ⚠️  진행 가능하나 결과 해석 시 주의')
    print(f'  현재: {pct}% → {"진행 가능 (주의)" if pct < 50 else "재실험 권장"}')
    if pct >= 50:
        raise AssertionError(
            f'fallback {pct}% ≥ 50%. 모델 출력 형식에 심각한 문제가 있습니다. '
            '위의 패치 셀을 다시 실행한 뒤 이 셀을 재실행하세요.'
        )
    else:
        print(f'\n→ {pct}% < 50%이므로 Phase 3~4를 계속 진행합니다.')
else:
    print('\n✅ 안정성 기준 통과. Phase 3~4 진행 가능.')

import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("GPU 메모리 해제 완료")

[Mistral-7B] turns=12, Blue actions=39, fallback=15(38%) ⚠️ 기준 미달
   15회  Missing required keys: actions, doctrine_reference, reasonin

T02 reasoning 샘플: Concentrate combat power on the decisive point, the urban crossroads (10,7), while maintaining security on the open flanks. The Red Force has no initial presence or visibility, so the maneuver must be

⚠️  fallback 기준 미달 — 원인별 대처 방법:
  • "Missing required keys" → extract_json_object 패치 필요 (위 패치 셀 확인)
  • "Unterminated JSON"     → --max-tokens를 4096으로 재시도
  • "No JSON found"        → 모델이 [INST] 템플릿 무시 — 정상 범위

📋 Phase 3~4 진행 판단 기준:
  fallback < 30%  → ✅ 진행 가능
  30% ≤ fallback < 50%  → ⚠️  진행 가능하나 결과 해석 시 주의
  현재: 38% → 진행 가능 (주의)

→ 38% < 50%이므로 Phase 3~4를 계속 진행합니다.


In [ ]:
# Llama 단일 게임 안정성 확인
!python scripts/run_single_game.py \
    --scenario s1_open_encounter \
    --blue-agent local_llm:meta-llama/Llama-3.1-8B-Instruct \
    --red-agent rule \
    --fog-preset llm \
    --max-tokens 1024 \
    --backend vllm \
    --output {RESULTS_BASE}/phase2/llama_stability_s1.jsonl

INFO: applied fog preset 'llm' to visibility_radius, identification_radius: visibility_radius=5, identification_radius=2.
2026-03-24 12:47:30.057961: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774356450.082621   15789 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774356450.090036   15789 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774356450.107378   15789 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774356450.107409   15789 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linki

In [ ]:
pct = analyze_stability(f'{RESULTS_BASE}/phase2/llama_stability_s1.jsonl', 'Llama-3.1-8B')
assert pct < 30, f'안정성 기준 미달({pct}%). Phase 3~4 진행 전 원인 파악 필요.'

import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("GPU 메모리 해제 완료. Section 3으로 진행하세요.")

[Llama-3.1-8B] turns=12, Blue actions=40, fallback=3(7%) ✅ 통과
    3회  Extracted JSON object is invalid: Expecting value.

T02 reasoning 샘플: Concentrate Blue Force at the crossroads to secure the position. Surprise the enemy with a rapid and unexpected arrival. Security is provided by Blue B's defensive posture.


In [ ]:
# Phase 4 시작 전 GPU 메모리 정리
# 안정성 테스트 이후 GPU 메모리가 남아 있을 수 있음
import gc, torch
gc.collect()
torch.cuda.empty_cache()

free_mem = torch.cuda.mem_get_info()[0] / 1024**3
total_mem = torch.cuda.mem_get_info()[1] / 1024**3
print(f"GPU 여유 메모리: {free_mem:.1f} GiB / {total_mem:.1f} GiB")
if free_mem < 20:
    print("⚠️  여유 메모리 부족. 런타임 재시작(Runtime > Restart runtime) 후 Section 0, 3만 실행하세요.")
else:
    print("✅ 메모리 충분. Phase 4 진행 가능.")


## 3. Phase 4-2 — Mistral-7B × 5시나리오 × 100회

> **예상 소요**: 시나리오당 약 50분~1시간 (100회 × 12턴 × ~5초/턴)
>
> **⚠️ 안정성 테스트 통과 후 실행하세요 (fallback < 30%).**
>
> 세션 단절 대비: 시나리오별로 셀이 분리되어 있어 중단 후 재개 가능.

In [ ]:
import os
SCENARIOS = [
    's1_open_encounter',
    's2_mountain_assault',
    's3_urban_fight',
    's4_river_crossing',
    's5_breakout',
]
for s in SCENARIOS:
    os.makedirs(f'{RESULTS_BASE}/phase4/mistral/{s}', exist_ok=True)
print('디렉토리 생성 완료')

디렉토리 생성 완료


In [3]:
# S1 — 평지 조우전
!python scripts/run_batch.py \
    --scenario s1_open_encounter \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 100 \
    --fog-preset llm \
    --max-tokens 1024 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s1_open_encounter
print('[Mistral] S1 완료')

INFO: applied fog preset 'llm' to visibility_radius, identification_radius: visibility_radius=5, identification_radius=2.
{
  "blue_win_rate": 0.48,
  "mean_action_entropy": 1.5586454658512243,
  "mean_escalation_sensitivity_index": 0.03246212121212121,
  "output_dir": "/content/drive/MyDrive/wargame_runs/phase4/mistral/s1_open_encounter",
  "red_win_rate": 0.5,
  "run_count": 100
}
[Mistral] S1 완료


In [4]:
# S2 — 산악 강습
!python scripts/run_batch.py \
    --scenario s2_mountain_assault \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 100 \
    --fog-preset llm \
    --max-tokens 1024 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s2_mountain_assault
print('[Mistral] S2 완료')

INFO: applied fog preset 'llm' to visibility_radius, identification_radius: visibility_radius=5, identification_radius=2.
{
  "blue_win_rate": 1.0,
  "mean_action_entropy": 0.0,
  "mean_escalation_sensitivity_index": 0.0,
  "output_dir": "/content/drive/MyDrive/wargame_runs/phase4/mistral/s2_mountain_assault",
  "red_win_rate": 0.0,
  "run_count": 100
}
[Mistral] S2 완료


In [5]:
# S3 — 시가지 전투
!python scripts/run_batch.py \
    --scenario s3_urban_fight \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 100 \
    --fog-preset llm \
    --max-tokens 1024 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s3_urban_fight
print('[Mistral] S3 완료')

INFO: applied fog preset 'llm' to visibility_radius, identification_radius: visibility_radius=5, identification_radius=2.
{
  "blue_win_rate": 0.23,
  "mean_action_entropy": 1.5234688411189714,
  "mean_escalation_sensitivity_index": 0.07212121212121211,
  "output_dir": "/content/drive/MyDrive/wargame_runs/phase4/mistral/s3_urban_fight",
  "red_win_rate": 0.77,
  "run_count": 100
}
[Mistral] S3 완료


In [6]:
# S4 — 하천 도하
!python scripts/run_batch.py \
    --scenario s4_river_crossing \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 100 \
    --fog-preset llm \
    --max-tokens 1024 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s4_river_crossing
print('[Mistral] S4 완료')

INFO: applied fog preset 'llm' to visibility_radius, identification_radius: visibility_radius=5, identification_radius=2.
{
  "blue_win_rate": 1.0,
  "mean_action_entropy": 0.0,
  "mean_escalation_sensitivity_index": 0.0,
  "output_dir": "/content/drive/MyDrive/wargame_runs/phase4/mistral/s4_river_crossing",
  "red_win_rate": 0.0,
  "run_count": 100
}
[Mistral] S4 완료


In [7]:
# S5 — 포위 돌파
!python scripts/run_batch.py \
    --scenario s5_breakout \
    --matchup 'local_llm:mistralai/Mistral-7B-Instruct-v0.3,rule' \
    --seed-count 100 \
    --fog-preset llm \
    --max-tokens 1024 \
    --backend vllm \
    --stochastic-combat \
    --noise-std 0.1 \
    --output-dir {RESULTS_BASE}/phase4/mistral/s5_breakout
print('[Mistral] S5 완료')

INFO: applied fog preset 'llm' to visibility_radius, identification_radius: visibility_radius=5, identification_radius=2.
{
  "blue_win_rate": 1.0,
  "mean_action_entropy": 0.48977982334345077,
  "mean_escalation_sensitivity_index": 0.020694444444444442,
  "output_dir": "/content/drive/MyDrive/wargame_runs/phase4/mistral/s5_breakout",
  "red_win_rate": 0.0,
  "run_count": 100
}
[Mistral] S5 완료


In [8]:
# Mistral 전체 결과 검증 (액션 레벨 fallback 집계)
import json, pathlib

base = pathlib.Path(f'{RESULTS_BASE}/phase4/mistral')
logs = list(base.rglob('*.jsonl'))
print(f'총 로그 파일: {len(logs)}개 (기대값: 500)')

total_blue = fb_blue = 0
blue_wins = red_wins = 0
per_scenario = {}

for p in logs:
    scenario = p.parent.name
    try:
        recs = [json.loads(l) for l in p.read_text().strip().split('\n') if l.strip()]
    except Exception:
        continue
    if not recs:
        continue

    per_scenario.setdefault(scenario, {'total': 0, 'fb': 0, 'games': 0})
    per_scenario[scenario]['games'] += 1

    for r in recs:
        for a in r.get('actions', []):
            if not a.get('unit_id', '').startswith('blue'):
                continue
            total_blue += 1
            per_scenario[scenario]['total'] += 1
            if a.get('metadata', {}).get('fallback'):
                fb_blue += 1
                per_scenario[scenario]['fb'] += 1

print()
print(f'{"시나리오":<25} {"게임":>5} {"Blue fallback":>14}')
print('-' * 48)
for s, v in sorted(per_scenario.items()):
    pct = 100 * v['fb'] // max(v['total'], 1)
    flag = ' ✅' if pct < 30 else ' ⚠️'
    print(f'{s:<25} {v["games"]:>5} {v["fb"]:>4}/{v["total"]:>5} ({pct:>3}%){flag}')
print()
overall = 100 * fb_blue // max(total_blue, 1)
print(f'전체 Blue fallback: {fb_blue}/{total_blue} ({overall}%)')
print(f'판정: {"✅ Phase 5 분석 가능" if overall < 30 else "⚠️ 재실험 필요"}')

총 로그 파일: 500개 (기대값: 500)

시나리오                         게임  Blue fallback
------------------------------------------------
Mistral-7B-Instruct-v0.3-vs-rule   500 19500/19500 (100%) ⚠️

전체 Blue fallback: 19500/19500 (100%)
판정: ⚠️ 재실험 필요


## 4. Phase 4-3 — Llama-3.1-8B × 5시나리오 × 100회

> **예상 소요**: Mistral과 동일, 총 약 4~5시간
>
> 새 Colab 세션에서 실행 권장 (GPU 메모리 초기화)

In [9]:
import os
SCENARIOS = [
    "s1_open_encounter", "s2_mountain_assault", "s3_urban_fight",
    "s4_river_crossing", "s5_breakout",
]
for s in SCENARIOS:
    os.makedirs(f"{RESULTS_BASE}/phase4/llama/{s}", exist_ok=True)

# HF 토큰: Colab 왼쪽 사이드바 🔑 (Secrets) 에서 불러옴
# 설정 방법: 사이드바 → Secrets → "HF_TOKEN" 추가
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("준비 완료 (HF_TOKEN 로드됨)")

준비 완료 (HF_TOKEN 로드됨)


In [10]:
# Llama 5시나리오 순차 실행
import subprocess

for scenario in SCENARIOS:
    print(f'[Llama] 시작: {scenario}')
    result = subprocess.run([
        'python', 'scripts/run_batch.py',
        '--scenario', scenario,
        '--matchup', 'local_llm:meta-llama/Llama-3.1-8B-Instruct,rule',
        '--seed-count', '100',
        '--fog-preset', 'llm',
        '--max-tokens', '1024',
        '--backend', 'vllm',
        '--stochastic-combat',
        '--noise-std', '0.1',
        '--output-dir', f'{RESULTS_BASE}/phase4/llama/{scenario}',
    ], capture_output=True, text=True)
    if result.returncode != 0:
        print(f'  ❌ ERROR: {result.stderr[-500:]}')
    else:
        # 마지막 줄 (JSON 요약) 출력
        last_line = [l for l in result.stdout.strip().split('\n') if l.strip()][-1]
        print(f'  ✅ 완료: {last_line}')

[Llama] 시작: s1_open_encounter
  ✅ 완료: }
[Llama] 시작: s2_mountain_assault
  ✅ 완료: }
[Llama] 시작: s3_urban_fight
  ✅ 완료: }
[Llama] 시작: s4_river_crossing
  ✅ 완료: }
[Llama] 시작: s5_breakout
  ✅ 완료: }


In [11]:
# Llama 결과 검증
import json, pathlib

base = pathlib.Path(f'{RESULTS_BASE}/phase4/llama')
logs = list(base.rglob('*.jsonl'))
print(f'총 로그 파일: {len(logs)}개 (기대값: 500)')

total_blue = fb_blue = 0
for p in logs:
    try:
        recs = [json.loads(l) for l in p.read_text().strip().split('\n') if l.strip()]
        for r in recs:
            for a in r.get('actions', []):
                if a.get('unit_id', '').startswith('blue'):
                    total_blue += 1
                    if a.get('metadata', {}).get('fallback'):
                        fb_blue += 1
    except Exception:
        pass

overall = 100 * fb_blue // max(total_blue, 1)
print(f'전체 Blue fallback: {fb_blue}/{total_blue} ({overall}%)')
print(f'판정: {"✅ 분석 가능" if overall < 30 else "⚠️ 재실험 필요"}')

총 로그 파일: 500개 (기대값: 500)
전체 Blue fallback: 19500/19500 (100%)
판정: ⚠️ 재실험 필요


## 5. 결과 취합 + 통계 분석

**전제**: Phase 3 베이스라인(rule-vs-rule, 250게임)은 로컬에서 완료됨.
이 셀은 Colab에서 얻은 LLM 결과만 분석합니다.
최종 통계 비교(LLM vs 베이스라인)는 결과를 로컬로 복사한 후 수행합니다.

In [12]:
import json, pathlib, statistics, math
import sys
sys.path.insert(0, 'src')

from wargame.analysis import (
    action_entropy,
    doctrine_compliance_rate,
    tactical_rationality_score,
    escalation_sensitivity_index,
    json_parsing_success_rate,
    win_rate,
    load_jsonl_records,
)
from wargame.core.enums import Faction

base = pathlib.Path(RESULTS_BASE)

GROUPS = {
    'Mistral-7B': list((base / 'phase4/mistral').rglob('*.jsonl')),
    'Llama-3.1-8B': list((base / 'phase4/llama').rglob('*.jsonl')),
}

print(f'{"에이전트":<15} {"게임수":>6} {"Blue승률":>9} {"Entropy":>9} {"DCR":>8} {"TRS":>8} {"파싱률":>8}')
print('-' * 70)

for name, logs in GROUPS.items():
    if not logs:
        print(f'{name:<15} (데이터 없음)')
        continue
    wr = win_rate(logs, faction=Faction.BLUE)
    ent = statistics.mean(action_entropy(p) for p in logs)
    dcr = statistics.mean(doctrine_compliance_rate(p) for p in logs)
    trs = statistics.mean(tactical_rationality_score(p) for p in logs)
    psr = statistics.mean(json_parsing_success_rate(p) for p in logs)
    print(f'{name:<15} {len(logs):>6} {wr:>9.3f} {ent:>9.3f} {dcr:>8.3f} {trs:>8.3f} {psr:>8.3f}')

print()
print('로컬 베이스라인 (Phase 3 참조값):')
print(f'  Rule-vs-Rule: Blue승률=0.672, Entropy=1.620, DCR=0.804, TRS=3.548')

에이전트               게임수    Blue승률   Entropy      DCR      TRS      파싱률
----------------------------------------------------------------------
Mistral-7B         500     0.742     0.714    0.889    3.937    0.000
Llama-3.1-8B       500     0.742     0.714    0.889    3.937    0.000

로컬 베이스라인 (Phase 3 참조값):
  Rule-vs-Rule: Blue승률=0.672, Entropy=1.620, DCR=0.804, TRS=3.548


In [13]:
# RQ1: LLM DCR > 0.5 (one-sample t-test + Cohen's d)
import numpy as np
from scipy import stats

def collect_dcr_scores(log_dir):
    """White Cell doctrine_compliance 점수 수집"""
    scores = []
    for p in pathlib.Path(log_dir).rglob('*.jsonl'):
        for rec in load_jsonl_records(p):
            wc = rec.get('metadata', {}).get('white_cell', {})
            s = wc.get('metadata', {}).get('scores', {}).get('doctrine_compliance')
            if s is not None:
                scores.append(float(s))
    return scores

print('=== RQ1: DCR > 0.5 One-sample t-test ===')
for model, log_dir in [
    ('Mistral-7B',   f'{RESULTS_BASE}/phase4/mistral/'),
    ('Llama-3.1-8B', f'{RESULTS_BASE}/phase4/llama/'),
]:
    dcr = collect_dcr_scores(log_dir)
    if not dcr:
        print(f'{model}: 데이터 없음')
        continue
    t_stat, p_val = stats.ttest_1samp(dcr, popmean=0.5)
    cohen_d = (np.mean(dcr) - 0.5) / np.std(dcr, ddof=1)
    sig = '★ 유의' if p_val < 0.05 else '비유의'
    print(f'{model}: n={len(dcr)}, mean={np.mean(dcr):.3f}, t={t_stat:.3f}, p={p_val:.4f}, d={cohen_d:.3f} [{sig}]')

=== RQ1: DCR > 0.5 One-sample t-test ===
Mistral-7B: n=6500, mean=0.890, t=1451.499, p=0.0000, d=18.004 [★ 유의]
Llama-3.1-8B: n=6500, mean=0.890, t=1451.499, p=0.0000, d=18.004 [★ 유의]


In [14]:
# RQ3: Action Entropy 비교 (Kruskal-Wallis)
from scipy import stats as scipy_stats

print('=== RQ3: Action Entropy Kruskal-Wallis ===')
entropy_groups = {}
for name, logs in GROUPS.items():
    if logs:
        vals = [action_entropy(p) for p in logs]
        entropy_groups[name] = vals
        print(f'  {name:<15}: mean={np.mean(vals):.3f} ± {np.std(vals):.3f} (n={len(vals)})')

# 로컬 베이스라인 참조값 추가 (상수)
print(f'  {"Rule (local)":<15}: mean=1.620 (n=250, Phase 3)')

if len(entropy_groups) >= 2:
    h, p_val = scipy_stats.kruskal(*entropy_groups.values())
    sig = '★ 유의' if p_val < 0.05 else '비유의'
    print(f'\nKruskal-Wallis (LLM 모델 간): H={h:.3f}, p={p_val:.4f} [{sig}]')

=== RQ3: Action Entropy Kruskal-Wallis ===
  Mistral-7B     : mean=0.714 ± 0.701 (n=500)
  Llama-3.1-8B   : mean=0.714 ± 0.701 (n=500)
  Rule (local)   : mean=1.620 (n=250, Phase 3)

Kruskal-Wallis (LLM 모델 간): H=0.000, p=1.0000 [비유의]


## 6. 결과 다운로드 (Colab → 로컬 Mac)

최종 통계 분석(LLM vs 베이스라인 비교)은 로컬에서 수행합니다.
아래 방법 중 하나로 결과를 Mac으로 복사하세요.

**방법 A: Google Drive 동기화** (Drive에 저장했다면 Mac에서 자동 동기화됨)
```bash
# Mac Terminal에서 실행
cp -r ~/Google\ Drive/My\ Drive/wargame_runs/phase4 \
  ~/Multi-Agent_Wargame/runs/
```

**방법 B: zip 다운로드**

In [15]:
# 결과를 zip으로 압축하여 다운로드
import shutil
from google.colab import files

zip_path = '/content/phase4_results'
shutil.make_archive(zip_path, 'zip', f'{RESULTS_BASE}/phase4')
print(f'압축 완료: {zip_path}.zip')
files.download(f'{zip_path}.zip')

압축 완료: /content/phase4_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. 로컬에서 실행할 최종 분석 명령어

Colab 결과를 `runs/phase4/`에 복사한 후 Mac Terminal에서 실행하세요:

```bash
# Phase 5-1: 통계 검정
python scripts/run_statistical_tests.py \
  --llm-dirs runs/phase4/mistral/ runs/phase4/llama/ \
  --baseline-dirs runs/phase3/baseline/ \
  --output runs/phase5/statistical_results.json

# Phase 5-2: 시각화
python scripts/generate_plots.py \
  --input-dirs runs/phase4/mistral/ runs/phase4/llama/ runs/phase3/baseline/ \
  --labels "Mistral-7B" "Llama-3.1-8B" "Rule-Based" \
  --output-dir runs/phase5/plots/

# Phase 5-3: 전체 지표 재집계
python scripts/evaluate_logs.py \
  runs/phase4/mistral/ runs/phase4/llama/ runs/phase3/baseline/
```

In [16]:
cp -r ~/Library/CloudStorage/GoogleDriveFilerStream/My\ Drive/wargame_runs/phase4/mistral \
  ~/Multi-Agent_Wargame/runs/phase4/

SyntaxError: invalid syntax (1757982446.py, line 1)